In [ ]:
# SETUP
# DICOM/RTSTRUCT to paired NRRD converter for Google Colab
#
# WHAT GOES IN
# A DICOM scan is not one file but a folder of many files, usually one file
# per image slice. The tumour outline (RTSTRUCT) is another file in the same
# patient folder.
# A "root" folder is the top-level folder you point the notebook to. The
# notebook searches it and every folder inside it.
#
# SETTINGS TO EDIT
# - Runtime: choose a high-RAM runtime (>50 GB working memory recommended).
# - dicom_root, dicom_root2, dicom_root3: Google Drive paths of up to three root
#   folders. All roots found are processed together. Unused or missing roots
#   are ignored. Patient folder names must be unique across all roots.
#   Google Drive folders with many files can make RAM usage spike to 30+ GB
#   for several minutes while the files are first read. To reduce this, spread
#   the patient folders over the three roots, always moving whole patient
#   folders.
# - output_root: folder for the converted paired NRRD image and mask files.
#   Notebook 2 INPUT_ROOT must use the
#   same path. Each tumour outline produces two files: the scan
#   (<patient>_<sequence>.nrrd) and its tumour mask, an image marking the
#   tumour (<patient>_<sequence>_mask.nrrd). Scans without a usable outline
#   are saved in a separate folder, created automatically as
#   <output_root>_unpaired. Notebook 2 needs one scan-mask pair per patient.
# - MENINGIOMA_KEYWORDS and SHORT_EXACT_CODES: an RTSTRUCT holds several named
#   outlines (ROIs, "regions of interest"). Check the names in 3D Slicer and
#   list the tumour names here, in lowercase:
#     KEYWORDS match anywhere in the name ("menin" matches "Meningioma_L").
#     SHORT_EXACT_CODES must equal the whole name ("tu" matches "TU" only).
#   Outlines with "skull" in the name are always excluded. All matching
#   outlines are merged into one mask. Check every generated mask visually.



# Install dependencies
!pip -q install pynrrd pydicom SimpleITK rt-utils numpy

import os, warnings, shutil, tempfile
import numpy as np
import pydicom
import SimpleITK as sitk
from rt_utils import RTStructBuilder

# Reduce SimpleITK log noise
sitk.ProcessObject_SetGlobalWarningDisplay(False)

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Paths: edit before running
dicom_root  = '/content/drive/MyDrive/DICOM_files_1'
dicom_root2  = '/content/drive/MyDrive/DICOM_files_2'
dicom_root3  = '/content/drive/MyDrive/DICOM_files_3'
output_root = '/content/drive/MyDrive/DATA/images_and_masks'
output_root_unpaired = output_root.rstrip('/\\') + '_unpaired'
os.makedirs(output_root, exist_ok=True)
os.makedirs(output_root_unpaired, exist_ok=True)
print(f"Paired NRRDs: {output_root}")
print(f"Image-only NRRDs: {output_root_unpaired}")
legacy_masks = [f for f in os.listdir(output_root) if f.lower().endswith('mask.seg.nrrd')]
if legacy_masks:
    warnings.warn(
        f'{len(legacy_masks)} legacy MASK.seg.nrrd file(s) already exist in output_root. '
        'Move or remove them before Notebook 2 to avoid duplicate masks.'
    )

# Edit these ROI names after inspecting representative studies in 3D Slicer.
MENINGIOMA_KEYWORDS = [
    "meningioma", "meningima", "mening2", "mening", "menin",
    "menmri", "menig", "tumor", "tum ", "adenoma", "herwin",
    "new volume menin", "postop", "men wrh", "men t1cc",
    "men po", "target",
]
SHORT_EXACT_CODES = {
    "m", "me", "men", "mg", "men2", "tu", "tum",
    "men po", "men wrh", "men t1cc", "tu pp", "tu2",
}

def is_target_roi(name: str) -> bool:
    n = str(name or '').strip().lower()
    return bool(n) and 'skull' not in n and (
        n in SHORT_EXACT_CODES or any(term in n for term in MENINGIOMA_KEYWORDS)
    )

# Helpers

# Use all available CPU threads inside ITK/SimpleITK ops
sitk.ProcessObject_SetGlobalDefaultNumberOfThreads(max(1, os.cpu_count() or 1))

def list_all_dicoms(root_dir):
    """Find DICOM files, including files without a .dcm extension."""
    out = []
    for dp, _, files in os.walk(root_dir):
        for f in files:
            fp = os.path.join(dp, f)
            if f.lower().endswith(".dcm"):
                out.append(fp)
                continue
            try:
                ds = pydicom.dcmread(fp, stop_before_pixels=True, force=True)
                if getattr(ds, "SOPClassUID", None):
                    out.append(fp)
            except Exception:
                pass
    return out

def get_tags_quick(path):
    ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
    return {
        "SOPInstanceUID": getattr(ds, "SOPInstanceUID", None),
        "SeriesInstanceUID": getattr(ds, "SeriesInstanceUID", None),
        "InstanceNumber": getattr(ds, "InstanceNumber", None),
        "ImagePositionPatient": getattr(ds, "ImagePositionPatient", None),
        "ImageOrientationPatient": getattr(ds, "ImageOrientationPatient", None),
        "Modality": getattr(ds, "Modality", None),
        "SeriesDescription": getattr(ds, "SeriesDescription", None),
    }

def index_series_files_by_uid(all_dcms):
    series_map = {}
    for fp in all_dcms:
        try:
            uid = get_tags_quick(fp)["SeriesInstanceUID"]
            if uid:
                series_map.setdefault(uid, []).append(fp)
        except Exception:
            continue
    for uid in list(series_map.keys()):
        series_map[uid] = sorted(series_map[uid])
    return series_map

def sort_slice_files(files):
    """Sort slices by position, then instance number and filename."""
    records = []
    for fp in files:
        try:
            ds = pydicom.dcmread(fp, stop_before_pixels=True, force=True)
            ipp = getattr(ds, "ImagePositionPatient", None)
            iop = getattr(ds, "ImageOrientationPatient", None)
            inst = getattr(ds, "InstanceNumber", None)

            if iop and len(iop) == 6 and ipp is not None:
                row = np.array(iop[0:3], dtype=float)
                col = np.array(iop[3:6], dtype=float)
                normal = np.cross(row, col)
                t = float(np.dot(np.array(ipp, dtype=float), normal))
            else:
                t = None

            records.append((fp, t, inst))
        except Exception:
            records.append((fp, None, None))

    if any(r[1] is not None for r in records):
        records.sort(key=lambda r: (float('inf') if r[1] is None else r[1]))
    elif any(r[2] is not None for r in records):
        records.sort(key=lambda r: (float('inf') if r[2] is None else r[2]))
    else:
        records.sort(key=lambda r: r[0])

    return [r[0] for r in records]

def load_sitk_from_files(files_sorted):
    if not files_sorted:
        raise RuntimeError("Empty file list for volume load.")
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(files_sorted)
    return reader.Execute()

def _save_nrrd(image_sitk, path, compress=True):
    writer = sitk.ImageFileWriter()
    writer.SetFileName(path)
    writer.SetImageIO("NrrdImageIO")
    writer.UseCompressionOn() if compress else writer.UseCompressionOff()
    writer.Execute(image_sitk)
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        raise RuntimeError(f"File not written or empty: {path}")

def is_rtstruct(path):
    try:
        ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
        return (getattr(ds, "SOPClassUID", "") == "1.2.840.10008.5.1.4.1.1.481.3") or \
               (getattr(ds, "Modality", "") == "RTSTRUCT")
    except Exception:
        return False

def list_rtstruct_files(root):
    """Find RTSTRUCT files, regardless of extension."""
    out = []
    for dp, _, files in os.walk(root):
        for f in files:
            fp = os.path.join(dp, f)
            if is_rtstruct(fp):
                out.append(fp)
    return sorted(out)

def get_referenced_series_uids_from_rtstruct(rtstruct_path):
    try:
        ds = pydicom.dcmread(rtstruct_path, stop_before_pixels=True, force=True)
    except Exception as e:
        warnings.warn(f"Failed to read RTSTRUCT {rtstruct_path}: {e}")
        return set()
    out = set()
    try:
        for rfr in getattr(ds, "ReferencedFrameOfReferenceSequence", []):
            for rstud in getattr(rfr, "RTReferencedStudySequence", []):
                for rser in getattr(rstud, "RTReferencedSeriesSequence", []):
                    uid = getattr(rser, "SeriesInstanceUID", None)
                    if uid:
                        out.add(uid)
    except Exception:
        pass
    return out

def choose_uid_with_most_files(candidates, series_map):
    best_uid, best_n = None, -1
    for uid in candidates:
        n = len(series_map.get(uid, []))
        if n > best_n:
            best_uid, best_n = uid, n
    return best_uid

def choose_fallback_uid(series_map):
    best_uid, best_n = None, -1
    for uid, files in series_map.items():
        n = len(files)
        if n > best_n:
            best_uid, best_n = uid, n
    return best_uid

def numpy_to_sitk_like(ref_img: sitk.Image, arr: np.ndarray) -> sitk.Image:
    img = sitk.GetImageFromArray(arr)  # (z,y,x)
    img.SetSpacing(ref_img.GetSpacing())
    img.SetOrigin(ref_img.GetOrigin())
    img.SetDirection(ref_img.GetDirection())
    return img

def safe_base_name(patient_dir, rtstruct_path=None, files_sorted=None):
    """Build patient_sequence; use SeriesDescription when needed."""
    patient = os.path.basename(os.path.normpath(patient_dir))
    seq_part = None

    if rtstruct_path:
        seq_folder = os.path.basename(os.path.dirname(rtstruct_path))
        if seq_folder.startswith(patient + "_"):
            seq_part = seq_folder[len(patient) + 1:]
        else:
            seq_part = seq_folder

    if not seq_part and files_sorted:
        try:
            ds0 = pydicom.dcmread(files_sorted[0], stop_before_pixels=True, force=True)
            desc = getattr(ds0, "SeriesDescription", None)
            if desc:
                seq_part = str(desc).strip().replace(" ", "_")
        except Exception:
            pass

    return f"{patient}_{seq_part}" if seq_part else patient

def make_temp_series_dir(files_sorted):
    """Copy one DICOM series to a temporary directory."""
    tmpdir = tempfile.mkdtemp(prefix="rtseries_")
    for i, src in enumerate(files_sorted, 1):
        dst = os.path.join(tmpdir, f"{i:06d}.dcm")
        shutil.copy2(src, dst)
    return tmpdir

def align_mask_axes(mask_np, target_shape):
    """Match common mask axis orders to target (z, y, x)."""
    if mask_np.shape == target_shape:
        return mask_np, "identity"
    for perm in [(2,0,1), (2,1,0), (1,2,0), (0,2,1), (1,0,2)]:
        try:
            cand = np.transpose(mask_np, perm)
            if cand.shape == target_shape:
                return cand, f"transpose{perm}"
        except Exception:
            pass
    return None, None

# Main conversion

# Use every configured root that exists.
configured_dicom_roots = [dicom_root, dicom_root2, dicom_root3]
available_dicom_roots = []
seen_dicom_roots = set()

print("\nChecking configured DICOM folders:")
for folder_number, raw_root in enumerate(configured_dicom_roots, 1):
    raw_root = "" if raw_root is None else str(raw_root).strip()
    if not raw_root:
        print(f"  Folder {folder_number}: NOT SET (ignored)")
        continue

    root = os.path.abspath(os.path.expanduser(raw_root))
    if root in seen_dicom_roots:
        print(f"  Folder {folder_number}: DUPLICATE PATH (ignored) -> {root}")
        continue
    seen_dicom_roots.add(root)

    if os.path.isdir(root):
        available_dicom_roots.append(root)
        print(f"  Folder {folder_number}: FOUND -> {root}")
    else:
        print(f"  Folder {folder_number}: NOT FOUND (ignored) -> {root}")

if not available_dicom_roots:
    raise FileNotFoundError(
        "None of the three configured DICOM folders could be found. "
        "Check dicom_root, dicom_root2 and dicom_root3, then run again."
    )

print(f"\nStarting conversion with {len(available_dicom_roots)} available DICOM folder(s).")

patients = []
for root in available_dicom_roots:
    root_patients = sorted(
        os.path.join(root, d) for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d))
    )
    patients.extend(root_patients)
    print(f"  {root}: {len(root_patients)} patient folder(s) found")

patients = sorted(patients)
if not patients:
    raise FileNotFoundError(
        'The available DICOM roots contain no patient subfolders. '
        'Each root must contain one folder per patient.'
    )

patient_names = {}
for patient_dir in patients:
    patient_names.setdefault(os.path.basename(patient_dir), []).append(patient_dir)
duplicate_names = {k: v for k, v in patient_names.items() if len(v) > 1}
if duplicate_names:
    details = '; '.join(f'{k}: {v}' for k, v in sorted(duplicate_names.items()))
    raise ValueError(
        'Duplicate patient-folder names were found across DICOM roots. '
        f'Rename or merge them to prevent output collisions: {details}'
    )

print(f"Total patient folders queued for conversion: {len(patients)}")
paired_outputs = 0
unpaired_outputs = 0
paired_by_patient = {}
reserved_outputs = set()

for p_idx, patient_dir in enumerate(patients, 1):
    print(f"\n[{p_idx}/{len(patients)}] Patient: {patient_dir}")
    all_dcms      = list_all_dicoms(patient_dir)
    series_map    = index_series_files_by_uid(all_dcms)
    rtstruct_list = list_rtstruct_files(patient_dir)

    print(f"  Series: {len(series_map)} | RTSTRUCTs: {len(rtstruct_list)} | DICOM files: {len(all_dcms)}")

    # Without RTSTRUCT, keep the image outside Notebook 2's paired-input folder.
    if not rtstruct_list:
        warnings.warn(f"No RTSTRUCT found under {patient_dir}; exporting an unpaired image.")
        fb_uid = choose_fallback_uid(series_map)
        if fb_uid:
            try:
                files_sorted = sort_slice_files(series_map[fb_uid])
                img = load_sitk_from_files(files_sorted)
                base = safe_base_name(patient_dir, files_sorted=files_sorted)
                out_img = os.path.join(output_root_unpaired, f"{base}.nrrd")
                _save_nrrd(img, out_img, compress=True)
                unpaired_outputs += 1
                print(f"  Saved unpaired fallback image: {out_img}")
            except Exception as e:
                warnings.warn(f"Failed to export fallback image for {patient_dir}: {e}")
        else:
            warnings.warn(f"No DICOM series found in {patient_dir}.")
        continue

    # Process each RTSTRUCT.
    for rs_idx, rtstruct_path in enumerate(rtstruct_list, 1):
        print(f"  RTSTRUCT [{rs_idx}/{len(rtstruct_list)}]: {rtstruct_path}")

        # Match the RTSTRUCT to its image series.
        ref_uids = get_referenced_series_uids_from_rtstruct(rtstruct_path)
        chosen_uid = choose_uid_with_most_files(ref_uids, series_map) if ref_uids else None
        if chosen_uid is None:
            warnings.warn("  Referenced SeriesInstanceUID not found locally; using the largest series as fallback.")
            chosen_uid = choose_fallback_uid(series_map)
        if chosen_uid is None:
            warnings.warn("  No DICOM series found in patient; skipping this RTSTRUCT.")
            continue

        files = series_map.get(chosen_uid, [])
        if not files:
            warnings.warn(f"  No files for chosen series UID {chosen_uid}; skipping this RTSTRUCT.")
            continue

        files_sorted = sort_slice_files(files)

        # Load the image volume.
        try:
            img = load_sitk_from_files(files_sorted)
        except Exception as e:
            warnings.warn(f"  Failed to load image series (UID {chosen_uid}): {e}")
            continue

        # Isolate the selected series for rt-utils.
        tmp_series_dir = None
        try:
            tmp_series_dir = make_temp_series_dir(files_sorted)
            rtstruct = RTStructBuilder.create_from(
                dicom_series_path=tmp_series_dir,
                rt_struct_path=rtstruct_path
            )
        except Exception as e:
            warnings.warn(f"  RTStructBuilder failed on {rtstruct_path}: {e}")
            if tmp_series_dir and os.path.isdir(tmp_series_dir):
                shutil.rmtree(tmp_series_dir, ignore_errors=True)
            continue

        # Keep matching tumour ROIs and exclude skull ROIs.
        roi_names = rtstruct.get_roi_names() or []

        target_roi_names = [n for n in roi_names if is_target_roi(n)]

        if not target_roi_names:
            print("  No meningioma-like ROI name found (after excluding SKULL). "
                  "Mask will not be written for this RTSTRUCT.")


        # Build a binary tumour mask in image array order (z, y, x).
        arr_shape = sitk.GetArrayFromImage(img).shape  # (z, y, x)
        label_arr = np.zeros(arr_shape, dtype=np.uint16)

        if target_roi_names:
            for roi_name in target_roi_names:
                try:
                    roi_mask = rtstruct.get_roi_mask_by_name(roi_name)  # often (y,x,z)
                    if roi_mask is None:
                        print(f"    ROI '{roi_name}' returned None; skipping.")
                        continue
                    aligned_mask, how = align_mask_axes(roi_mask, arr_shape)
                    if aligned_mask is None:
                        warnings.warn(
                            f"    ROI '{roi_name}' mask shape {roi_mask.shape} "
                            f"could not be aligned to image shape {arr_shape}; skipping."
                        )
                        continue
                    if how != "identity":
                        print(f"    ROI '{roi_name}': applied {how} to match (z,y,x)")
                    label_arr[aligned_mask > 0] = 1
                    print(f"    Added MENINGIOMA ROI '{roi_name}' (label 1)")
                except Exception as e:
                    warnings.warn(f"    Failed to rasterize ROI '{roi_name}': {e}")
        else:
            print("  MENINGIOMA ROI not found; no mask will be written.")

        base = safe_base_name(patient_dir, rtstruct_path, files_sorted)
        if np.any(label_arr):
            out_img = os.path.join(output_root, f"{base}.nrrd")
            out_mask = os.path.join(output_root, f"{base}_mask.nrrd")
            output_keys = {os.path.abspath(out_img), os.path.abspath(out_mask)}
            if reserved_outputs & output_keys:
                warnings.warn(f"  Duplicate output name for {base}; skipping to prevent overwrite.")
            else:
                try:
                    mask_img = numpy_to_sitk_like(img, label_arr)
                    mask_img = sitk.Cast(mask_img, sitk.sitkUInt16)
                    _save_nrrd(img, out_img, compress=True)
                    _save_nrrd(mask_img, out_mask, compress=True)
                    reserved_outputs.update(output_keys)
                    paired_outputs += 1
                    patient_key = os.path.basename(os.path.normpath(patient_dir))
                    paired_by_patient[patient_key] = paired_by_patient.get(patient_key, 0) + 1
                    print(f"  Saved pair: {out_img} | {out_mask}")
                    print("  ROI labels:", {n: 1 for n in target_roi_names})
                except Exception as e:
                    warnings.warn(f"  Failed to save NRRD pair: {e}")
        else:
            out_img = os.path.join(output_root_unpaired, f"{base}.nrrd")
            try:
                _save_nrrd(img, out_img, compress=True)
                unpaired_outputs += 1
                print(f"  Empty or unmatched mask; saved image separately: {out_img}")
            except Exception as e:
                warnings.warn(f"  Failed to save unpaired image NRRD: {e}")

        # Remove the temporary series directory.
        if tmp_series_dir and os.path.isdir(tmp_series_dir):
            shutil.rmtree(tmp_series_dir, ignore_errors=True)

multiple_pairs = {k: v for k, v in paired_by_patient.items() if v > 1}
if multiple_pairs:
    warnings.warn(
        'Multiple image-mask pairs were saved for these patients: '
        f'{multiple_pairs}. Notebook 2 requires one unambiguous pair per patient.'
    )
print(f"\nDone: {paired_outputs} paired and {unpaired_outputs} image-only output(s).")